<p style="background-color: #fce4ec; color: #ff0070;margin:0;display:inline-block ;padding:.6rem;border-radius:.25rem;  font-size:1.7rem">My Other Notebook Using Neural Network  <a href="https://www.kaggle.com/code/danishyousuf19/binary-prediction-of-poisonous-mushrooms" style="font-size: 18px;
  letter-spacing: 2px;
  text-transform: uppercase;
  display: inline-block;
  text-align: center;
  font-weight: bold;
  padding: 0.7em 2em;
  border: 3px solid #FF0072;
  border-radius: 2px;
  box-shadow: 0 2px 10px rgba(0, 0, 0, 0.16), 0 3px 6px rgba(0, 0, 0, 0.1);
  color: #FF0072;
  text-decoration: none;
  transition: 0.3s ease all;
  background:ghostwhite;
    border-radius:.3rem;
  z-index: 1;">Click Me</a></p>

    

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import optuna
from sklearn.metrics import confusion_matrix,matthews_corrcoef
import scipy
import warnings
warnings.filterwarnings('ignore')

  ### <p style="background-color: #fdefff;color:#c12eff;display: inline-block;padding:.6rem;border-radius:.5rem;border: 1px solid #c059ff">Loading data</p>

In [ ]:
train_data = pd.read_csv(r"/kaggle/input/playground-series-s4e8/train.csv")
test_data = pd.read_csv(r"/kaggle/input/playground-series-s4e8/test.csv")
sample_submission_data = pd.read_csv(r"/kaggle/input/playground-series-s4e8/sample_submission.csv")

print("train_data :", train_data.shape)
print("test_data :", test_data.shape)
print("sample_submission_data :", sample_submission_data.shape)

## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">Basic Info about Data</p> 

In [ ]:
train_data.head()

In [ ]:
test_data.head()

In [ ]:
sample_submission_data.head()

In [ ]:
train_data.info()

In [ ]:
train_data.describe()

In [ ]:
print(train_data['class'].value_counts())
sns.countplot(x='class',data=train_data)
plt.xticks(rotation=60)
plt.show()

## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">Correlation Matrics</p> 

In [ ]:
df_dropped = train_data.dropna()
df_dropped = train_data.drop('id',axis=1)
df_encoded = df_dropped.apply(lambda x: pd.factorize(x)[0] if x.dtype == 'object' else x)

correlation_matrix = df_encoded.corr()

correlation_matrix
plt.figure(figsize=(20, 20))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">Checking Unique Categories
</p> 

In [ ]:

cate_col = train_data.select_dtypes(include=['object']).columns

# Find unique categories and their counts for each categorical column
unique_categories = {col: train_data[col].value_counts() for col in cate_col}

# Set the size of the overall figure
plt.figure(figsize=(15, len(cate_col) * 5))

# Plot the count of each unique category
for i, (col, counts) in enumerate(unique_categories.items(), 1):
    plt.subplot(len(cate_col), 1, i)
    sns.barplot(x=counts.index, y=counts.values, palette="viridis")
    plt.title(f"Count of unique categories in column '{col}'")
    plt.xlabel('Categories')
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()

plt.show()


  ### <p style="background-color: #fdefff;color:#c12eff;display: inline-block;padding:.6rem;border-radius:.5rem;border: 1px solid #c059ff">Percentage of Missing Values by Feature</p>

In [ ]:
def null_percent(df):
    per=((df.isnull().sum()/len(df))*100).round(4)
    return per
print("Nan Values in Train data")
print(null_percent(train_data))
print("Nan Values in Test data")
print(null_percent(test_data))

## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">Checking Feature Importance and Dropping Useless Columns
</p> 

In [ ]:
alpha = 0.05
values = {}

for col in train_data.columns:
    if col == "class":
        continue

    A, B = train_data[col], train_data["class"]

    dfObserved = pd.crosstab(A, B) 
    chi2, p, dof, expected = scipy.stats.chi2_contingency(dfObserved.values)
    values[col] = p
    if p < alpha:
        # Reject null hypothesis
        print("{} is important. (p = {})".format(col, p))
    else:
        # Accept null hypothesis
        print("{} is NOT important. (p = {})".format(col, p))

In [ ]:
train_data = train_data.drop(['id',"veil-color",'veil-type'], axis=1)
test_data = test_data.drop(['id',"veil-color",'veil-type'], axis=1)

## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">Handling NaN Values And Less Frequent Categories</p> 

In [ ]:
numeric_feats = ["cap-diameter", "stem-height", "stem-width"]
for feat in numeric_feats:
    median_value = train_data[feat].mode()[0]
    train_data[feat].fillna(median_value, inplace=True)
    test_data[feat].fillna(median_value, inplace=True)

In [ ]:
numeric_feats = ["cap-diameter", "stem-height", "stem-width"]
plt.figure(figsize=(15, 5))
for i, column in enumerate(numeric_feats, 1):
    plt.subplot(1, len(numeric_feats), i)
    sns.boxplot(x=train_data[column])
    plt.title(f'Box Plot of {column}')
plt.show()


In [ ]:
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.01)
    Q3 = data[column].quantile(0.99)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return data[(data[column] < lower_bound) | (data[column] > upper_bound)]

# Detect and drop outliers
for column in numeric_feats:
    outliers = detect_outliers_iqr(train_data, column)
    print(f"Number of outliers in {column}: {outliers.shape[0]}")


In [ ]:
def cap_outliers(data, column):
    Q1 = data[column].quantile(0.01)
    Q3 = data[column].quantile(0.99)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    data[column] = np.where(data[column] < lower_bound, lower_bound, data[column])
    data[column] = np.where(data[column] > upper_bound, upper_bound, data[column])

for column in numeric_feats:
    cap_outliers(train_data, column)
    cap_outliers(test_data, column)


In [ ]:
for column in numeric_feats:
    outliers = detect_outliers_iqr(train_data, column)
    print(f"Number of outliers in {column}: {outliers.shape[0]}")

In [ ]:
import pandas as pd

def cleaning(df):
    threshold = 100
    
    cat_feats = ["cap-shape", "cap-surface", "cap-color", "does-bruise-or-bleed", "gill-attachment",
                "gill-spacing", "gill-color", "stem-root", "stem-surface", "stem-color",
                "has-ring", "ring-type", "spore-print-color", "habitat", "season"]
    
    for feat in cat_feats:
        # Convert to category type if not already
        df[feat] = df[feat].astype('category')
        
        # Add 'missing' and 'noise' categories if not present
        if 'missing' not in df[feat].cat.categories:
            df[feat] = df[feat].cat.add_categories('missing')
        if 'noise' not in df[feat].cat.categories:
            df[feat] = df[feat].cat.add_categories('noise')
        
        # Replace NaN values with 'missing'
        df[feat] = df[feat].fillna('missing')
        
        # Replace infrequent categories with 'noise'
        counts = df[feat].value_counts(dropna=False)
        infrequent_categories = counts[counts < threshold].index
        df[feat] = df[feat].apply(lambda x: 'noise' if x in infrequent_categories else x)
    
    return df

# Example usage
train_data = cleaning(train_data)
test_data = cleaning(test_data)


In [ ]:

print("Nan Values in Train data")
print(null_percent(train_data))
print("Nan Values in Test data")
print(null_percent(test_data))

In [ ]:
cat_feats = ["cap-shape", "cap-surface", "cap-color", "does-bruise-or-bleed", "gill-attachment",
             "gill-spacing", "gill-color", "stem-root", "stem-surface", "stem-color",
              "has-ring", "ring-type", "spore-print-color", "habitat", "season"]
for feat in cat_feats:
    train_data[feat] = train_data[feat].astype('category')
for feat in cat_feats:
    test_data[feat] = test_data[feat].astype('category')

In [ ]:
train_data.info()

## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">Splitting Data</p> 

In [ ]:
X = train_data.drop(['class'], axis=1)
y = train_data['class']
X.shape,y.shape

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">Using XGBoost with optuna</p> 

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=2)

# Define the objective function for Optuna
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 2000),
        'max_depth': trial.suggest_int('max_depth', 10, 50),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.001, 0.3),
        'subsample': trial.suggest_uniform('subsample', 0.3, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.3, 1.0),
        'gamma': trial.suggest_loguniform('gamma', 1e-8, 1.0),
        'lambda': trial.suggest_loguniform('lambda', 1e-8, 10.0),
        'alpha': trial.suggest_loguniform('alpha', 1e-8, 10.0),
        'scale_pos_weight': trial.suggest_uniform('scale_pos_weight', 1.0, 10.0)
    }

    model = XGBClassifier(**params, use_label_encoder=False, eval_metric='logloss', enable_categorical=True,tree_method='hist',device= 'cuda',objective='multi:softmax',num_class=2 )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mcc = matthews_corrcoef(y_test, y_pred)
    trial.set_user_attr("mcc", mcc)
    return mcc

# Callback to print the MCC score for each trial
def print_mcc_callback(study, trial):
    mcc = trial.user_attrs["mcc"]
    print(f"Trial {trial.number}: MCC = {mcc}")

# Optimize hyperparameters with Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=33, callbacks=[print_mcc_callback])

# Get the best parameters
best_params = study.best_params
print(f"Best parameters: {best_params}")

## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">After running Above code the best param are :</p>
### <pre style="background-color: #fdefff;color:#c12eff;display: inline-block;padding:.6rem;border-radius:.5rem;border: 1px solid #c059ff"> Trial 14: MCC = 0.98502712418870508</pre>
### <pre style="background-color: #fdefff;color:#c12eff;display: inline-block;padding:.6rem;border-radius:.5rem;border: 1px solid #c059ff"> Best parameters: {'n_estimators': 288, 'max_depth': 35, 'learning_rate': 0.04964969702056722, 'subsample': 0.9976503206175568, 'colsample_bytree': 0.451300957115364, 'gamma': 0.8965084856869137, 'lambda': 0.03608752969236549, 'alpha': 6.33187159990702e-05, 'scale_pos_weight': 4.575979634910302}</pre>


In [ ]:
# best_params= {'n_estimators': 726, 'max_depth': 50, 'learning_rate': 0.012015656292102479, 'subsample': 0.5667665955529018, 'colsample_bytree': 0.46185371676261483, 'gamma': 2.0684057884288805e-06, 'lambda': 3.5355532695771448, 'alpha': 0.01997697798393198, 'scale_pos_weight': 8.538523371497066}

In [ ]:
model = XGBClassifier(**best_params,enable_categorical=True,tree_method='hist',device= 'cuda',objective='multi:softmax',num_class=2)
model = model.fit(X, y)

## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">Saving Submission file Using XGBoost</p>

In [ ]:
id_column = sample_submission_data.pop('id')

# Make predictions on the test data
y_test_pred = model.predict(test_data)
y_test_pred_binary = (y_test_pred > 0.5).astype(int)  

# Create the submission DataFrame
submission_df = pd.DataFrame({
    'id': id_column,
    'class': y_test_pred_binary
})

# Map the binary predictions to 'e' and 'p'
submission_df['class'] = np.where(submission_df['class'] == 1, 'p', 'e')

# Save the submission DataFrame to a CSV file
submission_df.to_csv('submission_xgb65.csv', index=False)
print("Submission file created: submission.csv")


### <p style="background-color: #fdefff;color:#c12eff;margin: 0;display: inline-block;padding:.4rem;border-radius:.5rem;border: 1px solid #c059ff">Please Upvote if you Really liked this</p>